# NB02. Base Model Training & BB SHAP

**목적**: 블랙박스 Teacher(LightGBM)를 학습하고, TreeSHAP으로 ground truth attribution을 산출한다.

**Dependencies**: NB01 (datasets.pkl)

**Outputs** (`outputs/NB02/`):
- `bb_model_{name}.txt`: LightGBM native model
- `bb_shap_{name}.npy`: TreeSHAP values (n_test, n_features), float32
- `bb_prob_{name}.npy`: P(default) on test, float32
- `bb_score_{name}.npy`: Integer credit score on test, int16
- `bb_meta_{name}.json`: HPs, AUC, best_iter, feature importance
- `train_val_idx_{name}.pkl`: train/val 내부 분할 인덱스 (surrogate 재사용)

**Design**:
- LightGBM unconstrained (no monotone) — 의도적. Surrogate 근사를 어렵게 하여 C1 효과 극대화
- 고정 HP (tuning 없음) — BB는 기여(contribution)가 아님
- TreeSHAP on full test set (subsampling 없음)
- Surrogate target: `transform_logit_to_score(prob)` = 정수 credit score (PDO=40, anchor=500)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle, json, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from decentra._utils import transform_logit_to_score, logit

RS = 42
NB01_DIR = Path('../outputs/NB01')
OUT_DIR = Path('../outputs/NB02')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (NB01_DIR / 'datasets.pkl').exists(), "Run NB01 first!"
with open(NB01_DIR / 'datasets.pkl', 'rb') as f:
    datasets = pickle.load(f)
print(f'Loaded: {list(datasets.keys())}')

Loaded: ['GMSC', 'HC']


## 1. Train & Evaluate

In [2]:
LGB_PARAMS = dict(
    max_depth=-1, n_estimators=1000, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, verbose=-1, random_state=RS, n_jobs=-1,
)

for name in datasets:
    d = datasets[name]
    X_train, X_test = d['X_train'], d['X_test']
    y_train, y_test = d['y_train'], d['y_test']
    
    # Internal train/val split for early stopping
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=RS
    )
    
    print(f'\n{"="*60}')
    print(f'  {name}: train={len(X_tr):,}, val={len(X_val):,}, test={len(X_test):,}')
    print(f'{"="*60}')
    
    # Train
    t0 = time.time()
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    train_time = time.time() - t0
    
    # Evaluate
    prob_test = model.predict_proba(X_test)[:, 1]
    auc_test = roc_auc_score(y_test, prob_test)
    auc_val = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    
    print(f'  AUC: val={auc_val:.4f}, test={auc_test:.4f}')
    print(f'  Best iter: {model.n_estimators_}, train time: {train_time:.0f}s')
    
    # BB TreeSHAP on full test set
    t0 = time.time()
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    shap_values = np.array(shap_values, dtype=np.float32)
    shap_time = time.time() - t0
    print(f'  SHAP: {shap_values.shape}, {shap_time:.0f}s')
    
    # Surrogate targets
    score_test = transform_logit_to_score(prob_test)
    score_train = transform_logit_to_score(model.predict_proba(X_train)[:, 1])
    
    # Feature importance (mean |SHAP|)
    feat_imp = pd.Series(
        np.abs(shap_values).mean(axis=0), 
        index=d['feature_names']
    ).sort_values(ascending=False)
    print(f'\n  Top-5 features (mean |SHAP|):')
    for fn, imp in feat_imp.head(5).items():
        print(f'    {fn}: {imp:.4f}')
    
    # Save artifacts
    model.booster_.save_model(str(OUT_DIR / f'bb_model_{name}.txt'))
    np.save(OUT_DIR / f'bb_shap_{name}.npy', shap_values)
    np.save(OUT_DIR / f'bb_prob_{name}.npy', prob_test.astype(np.float32))
    np.save(OUT_DIR / f'bb_score_test_{name}.npy', score_test.astype(np.int16))
    np.save(OUT_DIR / f'bb_score_train_{name}.npy', score_train.astype(np.int16))
    
    with open(OUT_DIR / f'bb_meta_{name}.json', 'w') as f:
        json.dump({
            'params': LGB_PARAMS,
            'best_iteration': int(model.n_estimators_),
            'auc_val': round(auc_val, 4),
            'auc_test': round(auc_test, 4),
            'train_time_s': round(train_time, 1),
            'shap_time_s': round(shap_time, 1),
            'n_train': len(X_train), 'n_test': len(X_test),
            'n_features': X_train.shape[1],
            'feature_names': d['feature_names'],
            'feature_importance': {fn: round(float(imp), 6) for fn, imp in feat_imp.items()},
            'class_dist_train': f'{y_train.mean():.4f}',
            'class_dist_test': f'{y_test.mean():.4f}',
        }, f, indent=2)
    
    # Save train/val indices for surrogate reuse
    with open(OUT_DIR / f'train_val_idx_{name}.pkl', 'wb') as f:
        pickle.dump({
            'train_idx': X_tr.index.tolist(),
            'val_idx': X_val.index.tolist(),
        }, f)
    
    print(f'\n  Artifacts saved to {OUT_DIR}/')


  GMSC: train=96,000, val=24,000, test=30,000
Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[90]	valid_0's binary_logloss: 0.177337
  AUC: val=0.8666, test=0.8649
  Best iter: 90, train time: 0s


  SHAP: (30000, 10), 3s

  Top-5 features (mean |SHAP|):
    RevolvingUtilizationOfUnsecuredLines: 0.6880
    NumberOfTime30-59DaysPastDueNotWorse: 0.3223
    age: 0.2324
    NumberOfTimes90DaysLate: 0.2125
    NumberOfOpenCreditLinesAndLoans: 0.1625

  Artifacts saved to ..\outputs\NB02/



  HC: train=196,806, val=49,202, test=61,503


Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[166]	valid_0's binary_logloss: 0.252023


  AUC: val=0.7340, test=0.7464
  Best iter: 166, train time: 3s


  SHAP: (61503, 41), 19s



  Top-5 features (mean |SHAP|):
    EXT_SOURCE_3: 0.3566
    EXT_SOURCE_2: 0.3356
    AMT_GOODS_PRICE: 0.1974
    AMT_CREDIT: 0.1356
    DAYS_EMPLOYED: 0.1131

  Artifacts saved to ..\outputs\NB02/


## 2. Summary

In [3]:
# File listing
print(f'Outputs ({OUT_DIR}):')
for p in sorted(OUT_DIR.iterdir()):
    s = p.stat().st_size
    label = f'{s/1024/1024:.1f} MB' if s > 1e6 else f'{s/1024:.0f} KB'
    print(f'  {p.name:40s} {label}')

Outputs (..\outputs\NB02):
  bb_meta_GMSC.json                        1 KB
  bb_meta_HC.json                          3 KB
  bb_model_GMSC.txt                        602 KB
  bb_model_HC.txt                          1.1 MB
  bb_prob_GMSC.npy                         117 KB
  bb_prob_HC.npy                           240 KB
  bb_score_test_GMSC.npy                   59 KB
  bb_score_test_HC.npy                     120 KB
  bb_score_train_GMSC.npy                  234 KB
  bb_score_train_HC.npy                    481 KB
  bb_shap_GMSC.npy                         1.1 MB
  bb_shap_HC.npy                           9.6 MB
  train_val_idx_GMSC.pkl                   483 KB
  train_val_idx_HC.pkl                     1.1 MB
